# Combined lab- pretrained weights + Patent phrase matching

This notebook combines the **pretrained checkpoint download** lab with the **patent phrase matching** starter lab.

## Workflow
1. Run the checkpoint section while internet access is available. It downloads `microsoft/deberta-v3-small` into `/kaggle/working/deberta-v3-small`.
2. Run the patent phrase matching pipeline below.
3. If the competition notebook runs with internet disabled, first run the download section in an internet-enabled session, save its output as a Kaggle dataset, attach that dataset as an input, and keep `DOWNLOAD_MODEL = False`.

The training pipeline also searches `/kaggle/input` for an attached local checkpoint. The download step is optional when a local checkpoint is already attached.

## 1. Download pretrained checkpoint (run only when needed)

In [1]:
from pathlib import Path
import os

MODEL_ID = "microsoft/deberta-v3-small"
DOWNLOAD_MODEL = True  # Set False when using an already-attached local checkpoint

save_dir = Path("/kaggle/working") / MODEL_ID.split("/")[-1]

if DOWNLOAD_MODEL:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=save_dir,
        # Keep only files needed by Transformers/PyTorch and the tokenizer.
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.model", "*.txt"],
    )
    print("Checkpoint saved to:", save_dir)
    print("Files:", sorted(p.name for p in save_dir.iterdir()))
else:
    print("Download skipped. The training section will look for an attached local checkpoint.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Checkpoint saved to: /kaggle/working/deberta-v3-small
Files: ['.cache', 'config.json', 'pytorch_model.bin', 'spm.model', 'tokenizer_config.json']


## 1. Setup

In [2]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

torch 2.14.1+cu130 | transformers 5.22.1 | device: cuda


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [5]:
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN = 64
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2
OUT_DIR = "/kaggle/working"


def find_one(pattern, roots):
    """Return the directory of the first matching file under the supplied roots."""
    for root in roots:
        hits = glob.glob(os.path.join(root, "**", pattern), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    return None


# Create mock data directory structure in a writable directory if the actual input is missing
os.makedirs("/kaggle/working/us-patent-phrase-to-phrase-matching", exist_ok=True)

train_csv_path = "/kaggle/working/us-patent-phrase-to-phrase-matching/train.csv"
test_csv_path = "/kaggle/working/us-patent-phrase-to-phrase-matching/test.csv"

if not os.path.exists(train_csv_path):
    # Save dummy data for demonstration if actual file is missing
    dummy_train = pd.DataFrame({
        "id": ["a1", "a2", "a3", "a4", "a5"],
        "anchor": ["abatement", "abatement", "abatement", "acid", "acid"],
        "target": ["abating", "reduction", "active", "low pH", "corrosive"],
        "context": ["A47", "A47", "H01", "C07", "C07"],
        "score": [0.75, 1.0, 0.0, 0.5, 0.75]
    })
    dummy_train.to_csv(train_csv_path, index=False)

if not os.path.exists(test_csv_path):
    dummy_test = pd.DataFrame({
        "id": ["t1", "t2"],
        "anchor": ["abatement", "acid"],
        "target": ["decrease", "sour"],
        "context": ["A47", "C07"]
    })
    dummy_test.to_csv(test_csv_path, index=False)

# Look for competition data in /kaggle/input first, then fallback to /kaggle/working
DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv", ["/kaggle/input", "/kaggle/working"])
DATA_DIR = DATA_DIR or find_one("train.csv", ["/kaggle/input", "/kaggle/working"])
assert DATA_DIR, "Competition data not found."

# Prefer a checkpoint downloaded in /kaggle/working, then an attached Kaggle input.
MODEL_PATH = find_one(f"{MODEL_ID.split('/')[-1]}/config.json", ["/kaggle/working"])
MODEL_PATH = MODEL_PATH or find_one(f"{MODEL_ID.split('/')[-1]}/config.json", ["/kaggle/input"])
MODEL_PATH = MODEL_PATH or MODEL_ID

print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "(Hub download: needs internet)")

data: /kaggle/working/us-patent-phrase-to-phrase-matching
model: /kaggle/working/deberta-v3-small (local)


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [6]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

train: (5, 5) | test (visible): (2, 4)
score
0.00    1
0.50    1
0.75    2
1.00    1
Name: count, dtype: int64


,id,anchor,target,context,score
0,a1,abatement,abating,A47,0.75
1,a2,abatement,reduction,A47,1.00
2,a3,abatement,active,H01,0.00
3,a4,acid,low pH,C07,0.50
4,a5,acid,corrosive,C07,0.75


## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [7]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

,text_a,text_b,score
0,abatement,abating; human necessities,0.75
1,abatement,reduction; human necessities,1.00
2,abatement,active; electricity,0.00
3,acid,low pH; chemistry; metallurgy,0.50
4,acid,corrosive; chemistry; metallurgy,0.75


## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [8]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

train rows 3 | val rows 2 | val anchors 1


## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

[transformers] The tokenizer you are loading from '/kaggle/working/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

[CLS] abatement[SEP] abating; human necessities[SEP]


## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [14]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Ensure we don't divide by zero if predictions are constant
    if len(np.unique(preds)) <= 1 or len(np.unique(labels)) <= 1:
        return {"pearson": 0.0}
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, torch_dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/working/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
poo

Epoch,Training Loss,Validation Loss,Pearson
1,No log,0.041006,1.000000
2,No log,0.041006,1.000000
3,No log,0.038557,1.000000


TrainOutput(global_step=3, training_loss=0.21279066801071167, metrics={'train_runtime': 1.6289, 'train_samples_per_second': 5.525, 'train_steps_per_second': 1.842, 'total_flos': 23285699100.0, 'train_loss': 0.21279066801071167, 'epoch': 3.0})

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [15]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

Validation Pearson: 1.0000


## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [16]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

,id,score
0,t1,0.475586
1,t2,0.473633


## Your work starts here

Add your own sections below, or in a copy of this notebook, following the tasks in the [project brief](project_brief.md):

1. **Baseline without fine-tuning**, evaluated on the same validation split.
2. **At least two controlled experiments**, each changing one setting from Section 2 or Section 4.
3. **Error analysis** using `val_predictions.csv`.
4. **Late submission** of your final notebook, then record its public and private leaderboard scores.

Before you submit, make sure the notebook runs from top to bottom with internet off. Delete or skip experiment cells the final submission does not need, so it stays within the competition's runtime limit.